# Ember · 01 · Aggregate FIRMS archives into monthly grid cells

Input: the Kaggle Dataset holding the FIRMS per-country yearly files (MODIS, VIIRS S-NPP, VIIRS NOAA-20).
Output: `hotspots_monthly.parquet`, one row per platform × country × year × month × 0.1° cell (schema in `docs/DATA_CONTRACT.md`).

Runs on a CPU kernel; the code lives in the GitHub repo so the notebook stays a thin runner.

In [ ]:
DATASET = "<kaggle-user>/<dataset-slug>"   # TODO: set once the dataset is online
REPO = "https://github.com/GAUSS-TPAC/Ember---Harmonization-of-MODIS-and-VIIRS-Hot-Spots.git"
RESOLUTION_DEG = 0.1
CHUNKSIZE = 2_000_000

In [ ]:
import subprocess, sys
subprocess.run(["git", "clone", "--depth", "1", REPO, "/kaggle/working/ember"], check=True)
sys.path.insert(0, "/kaggle/working/ember/pipeline")
from ember_pipeline.aggregate import aggregate_paths
from ember_pipeline.grid import Grid

In [ ]:
from pathlib import Path
root = Path("/kaggle/input") / DATASET.split("/")[-1]
inputs = sorted(p for p in root.rglob("*") if p.suffix.lower() in {".csv", ".zip", ".gz"})
print(len(inputs), "input files")
for p in inputs[:5]: print("  ", p.relative_to(root))

In [ ]:
import time
t0 = time.time()
done = []
def log(name):
    done.append(name)
    if len(done) % 50 == 0: print(f"{len(done)} files, {time.time()-t0:.0f}s", flush=True)
monthly = aggregate_paths(inputs, Grid(RESOLUTION_DEG), CHUNKSIZE, on_file=log)
print(f"{len(monthly):,} rows in {time.time()-t0:.0f}s")

In [ ]:
out = Path("/kaggle/working/hotspots_monthly.parquet")
monthly.to_parquet(out, index=False)
print(out, f"{out.stat().st_size/1e6:.1f} MB")

## Sanity check: the sensor jump this project exists to remove

In [ ]:
import matplotlib.pyplot as plt
yearly = monthly.groupby(["platform", "year"])["n"].sum().unstack(0)
ax = yearly.plot(figsize=(10, 4), logy=True, marker="o")
ax.set_ylabel("detections / year (log)"); ax.set_title("Raw detections per platform, world")
plt.show()
yearly.tail(15)